In [1]:
import json
import numpy as np
import os

dir_path = 'results_json'
result_files = os.listdir(dir_path)
result_paths = [os.path.join(dir_path, file) for file in result_files]

In [2]:
fold_num = 10
task_num = 4

for path in result_paths:
    with open(path, 'r', encoding='utf-8') as opened_file:
        data = json.load(opened_file)
        performance = data['performance']
        fold_acc, fold_mf1 = [], []
        for fold in range(fold_num):
            metrics_acc, metrics_mf1 = [], []
            for task in range(task_num + 1):
                metrics_acc.append(performance[f'task{task}_fold{fold}']['acc'])
                metrics_mf1.append(performance[f'task{task}_fold{fold}']['mF1'])
            fold_acc.append(metrics_acc)
            fold_mf1.append(metrics_mf1)
        fold_acc = np.array(fold_acc)[:, 1:, :]
        fold_mf1 = np.array(fold_mf1)[:, 1:, :]
        mu_acc, sigma_acc = np.mean(fold_acc, axis=0), np.std(fold_acc, axis=0)
        mu_mf1, sigma_mf1 = np.mean(fold_mf1, axis=0), np.std(fold_mf1, axis=0)
    result = {
        'R_acc_mean': mu_acc.tolist(),
        'R_acc_std': sigma_acc.tolist(),
        'R_mf1_mean': mu_mf1.tolist(),
        'R_mf1_std': sigma_mf1.tolist()
    }
    root, ext = os.path.splitext(path)
    new_path = f'{root}_table{ext}'
    with open(new_path, 'w', encoding='utf-8') as opend_file:
        json.dump(result, opend_file, indent=4)